# SOL: seeds 17 and 123, 500–4,000 examples, both Qwen models
Run cells in order in an allocated NVIDIA GPU Jupyter session. This queues 16 independent one-epoch training runs. The existing seed-42 trainer and wrapper are preserved byte-for-byte. Training uses the existing Python 3.12 environment; the notebook kernel can be newer.

Inputs: `/home/fchaudh3/Model Distillation`. Requires matching `release_manifest.json`, `formatting_contract.json`, all four training JSONLs and summaries, plus the existing original release/encoder on scratch. No dataset download or Drive connection is needed.

The uploaded training JSONLs must match both their release hashes and the recorded seed-42 run hashes. If they differ, stop and reconcile the data: a new seed alone should not change the dataset. This notebook does not bypass missing metadata or readiness checks; it retains the original preliminary experiment status.

Periodic checkpoints are saved every 100 updates. Rerunning the main queue resumes an interrupted run from its latest structurally complete checkpoint into a new segment folder, with the trainer checking exact configuration compatibility. Completed runs are skipped. Do not run two copies of the queue simultaneously.

Results: `/scratch/fchaudh3/capstone_person2_sol/results/sol_seeds17_123_upto4000/`. Handoff ZIPs: `/home/fchaudh3/capstone_person2_handoffs/`. The final index identifies the correct folder if a run was resumed.


## 1. Settings: 16 main runs, no seed-42 retraining


In [ ]:
from pathlib import Path
import os, sys, platform, getpass, subprocess, json, hashlib, shutil
USER = getpass.getuser()
SCRATCH = Path('/scratch')/USER
if not SCRATCH.is_dir():
    raise RuntimeError(f'{SCRATCH} is unavailable. Set SCRATCH to your actual SOL scratch/project directory.')
ROOT = SCRATCH/'capstone_person2_sol'
PACKAGE = ROOT/'code_seeds17_123'
DATA_ROOT = ROOT/'data'
RELEASE = DATA_ROOT/'person1_seeds17_123_verified'
UPLOADS = Path('/home/fchaudh3/Model Distillation')
ATTEMPT = 'sol_seeds17_123_upto4000'
WORK = ROOT/'results'/ATTEMPT
MODEL_SIZES = ['0.5B','1.5B']
DATASET_SIZES = [500, 1000, 2000, 4000]
SEEDS = [17, 123]
SEED = 17  # Smoke validation only; main loop uses SEEDS
SEED42_ROOTS = [ROOT/'results/sol_dual_v4_corrected01', ROOT/'results/sol_dual_v3_preliminary01']
READINESS = WORK/'team_readiness.json'
ENV = ROOT/'venv_torch260_cu124'
PYTHON = ENV/'bin/python'
HANDOFFS = Path.home()/'capstone_person2_handoffs'
WHO = 'Feni Chaudhari'
AUTO_BACKUP_TO_DRIVE = False
DRIVE_REMOTE = 'gdrive'
DRIVE_FOLDER = 'capstone_person2_SOL'
# Optional absolute rclone executable path if not in the Jupyter PATH:
RCLONE_OVERRIDE = '/packages/apps/rclone/1.68.1/bin/rclone'
for p in [ROOT,PACKAGE,DATA_ROOT,WORK,HANDOFFS]:p.mkdir(parents=True,exist_ok=True)
os.environ['HF_HOME'] = str(ROOT/'hf_cache')
os.environ['PIP_CACHE_DIR'] = str(ROOT/'pip_cache')
os.environ['OMP_NUM_THREADS'] = '4'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
print('Python:',sys.version.split()[0], 'architecture:',platform.machine())
print('Slurm job:',os.environ.get('SLURM_JOB_ID','NOT DETECTED'))
print('Results:',WORK)
assert os.environ.get('SLURM_JOB_ID'), 'Run in the allocated compute-node Jupyter session, not a login node.'
assert platform.machine()=='x86_64', 'This environment targets x86_64 NVIDIA nodes, not ARM/Gaudi/AMD.'
# The Jupyter kernel may use Python 3.14; training uses SOL's Python 3.12.
TRAINING_PYTHON = Path('/packages/apps/jupyter/2025-03-24/bin/python3.12')
assert TRAINING_PYTHON.is_file(), 'SOL Python 3.12 executable not found.'
subprocess.run([
    str(TRAINING_PYTHON), '-c',
    "import sys; print('Training Python:', sys.version); assert sys.version_info[:2] == (3, 12)"
], check=True)
subprocess.run(['nvidia-smi'],check=True)

# Let %%writefile save visible scripts in this directory.
os.chdir(PACKAGE)


## 2. Write the visible training engine
This file handles loading a frozen 4-bit base, adding LoRA, predicting the next solution token, backpropagation, optimizer updates, logging, saving, and reloading. You do not need to edit this long code cell. It is ordinary Python, not a token or encoded ZIP. Separate processes ensure only one model occupies the GPU at a time.

In [ ]:
%%writefile trainer_core.py
#!/usr/bin/env python3
"""Single-NVIDIA-GPU QLoRA pilot. See README before running real experiments.
Only standard-library imports are needed for --help, subset and unit tests.
GPU/model integration MUST be validated on the user's machine.
"""
import argparse
import collections
import csv
import hashlib
import importlib.metadata
import json
import math
import os
from pathlib import Path
import platform
import random
import subprocess
import sys
import time

MODEL = 'Qwen/Qwen2.5-0.5B'
REVISION = '060db6499f32faf8b98477b0a26969ef7d8b9987'
COUNT_FIELDS = ('prompt_tokens', 'completion_tokens', 'supervised_tokens', 'total_tokens')

def sha(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

def dump(path, obj):
    Path(path).write_text(json.dumps(obj, indent=2, ensure_ascii=False) + '\n', encoding='utf-8')

def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))

def load_rows(path):
    path = Path(path)
    if path.suffix == '.parquet':
        import pandas as pd
        rows = pd.read_parquet(path).to_dict('records')
    elif path.suffix == '.jsonl':
        rows = [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]
    else:
        raise ValueError('Use a self-contained .jsonl manifest or .parquet; ID-only text is insufficient.')
    if not rows:
        raise ValueError('Empty dataset')
    for r in rows:
        for field in ('row_id', 'problem_group_id', 'dataset_id', 'problem', 'completion', 'final_answer'):
            if field not in r or not isinstance(r[field], str) or not r[field].strip():
                raise ValueError(f'Missing/non-string/empty {field}; agree an explicit schema conversion with Person 1.')
        for field in COUNT_FIELDS:
            if field not in r or isinstance(r[field], bool) or int(r[field]) != r[field] or r[field] < 0:
                raise ValueError(f'Invalid/missing {field}: {r["row_id"]}')
    return rows

def source(r):
    name = r['dataset_id'].lower()
    if 'orca' in name:
        return 'orca'
    if 'numina' in name:
        return 'numina'
    raise ValueError(f'Unknown dataset_id {r["dataset_id"]!r}; adapt source() explicitly.')

def check_unique(rows, allow_repeats=False):
    # Intentional repeated exposures are permitted only with byte-equivalent row contents.
    ids, groups = {}, {}
    for r in rows:
        rid, gid = r['row_id'], r['problem_group_id']
        if gid in groups and groups[gid] != rid:
            raise ValueError(f'Two representatives of problem group {gid}')
        if rid in ids and (not allow_repeats or ids[rid] != r):
            raise ValueError(f'Duplicate/conflicting row_id {rid}')
        ids[rid], groups[gid] = r, rid

def load_contract(template_path, policy_path):
    # JSON preserves every newline and trailing space in Person 1's exact template.
    obj = read_json(template_path)
    if set(obj) != {'template'} or not isinstance(obj['template'], str):
        raise ValueError('Template JSON must contain exactly one string field: template')
    template = obj['template']
    if template.count('{problem}') != 1:
        raise ValueError('Template must contain exactly one literal {problem} placeholder.')
    policy = read_json(policy_path)
    required = {'confirmed_with_person1', 'add_special_tokens', 'append_eos', 'max_total_tokens', 'provenance'}
    if set(policy) != required or policy['confirmed_with_person1'] is not True or not policy['provenance']:
        raise ValueError('Obtain the exact formatting/tokenization policy from Person 1 and confirm it in JSON.')
    if policy['add_special_tokens'] is not False or policy['append_eos'] is not True:
        raise ValueError('This implementation supports add_special_tokens=false + one explicitly appended EOS. Reconcile other policies before use.')
    if not isinstance(policy['max_total_tokens'], int) or policy['max_total_tokens'] < 2:
        raise ValueError('Invalid context cap')
    return template, policy

def encode_row(r, tokenizer, template, cap):
    prompt = template.replace('{problem}', r['problem'])
    completion = r['completion']  # Never strip, rewrite final markers or add reasoning.
    pids = tokenizer.encode(prompt, add_special_tokens=False)
    joined = tokenizer.encode(prompt + completion, add_special_tokens=False)
    if not pids:
        raise ValueError(f'{r["row_id"]}: empty prompt token sequence')
    boundary_method = 'prefix'
    prompt_count = len(pids)
    if joined[:len(pids)] != pids:
        # Match Person 1 cell 25: full-text offsets may recover a valid boundary.
        offset_enc = tokenizer(prompt + completion, add_special_tokens=False, return_offsets_mapping=True)
        offsets = offset_enc['offset_mapping']
        boundary = len(prompt)
        if any(left < boundary < right for left, right in offsets):
            raise ValueError(f'{r["row_id"]}: token crosses prompt/completion boundary; inspect with Person 1.')
        prompt_count = sum(right <= boundary for left, right in offsets)
        joined = offset_enc['input_ids']
        if not 0 < prompt_count < len(joined):
            raise ValueError(f'{r["row_id"]}: invalid offset-based boundary')
        # Reject ambiguous/noncontiguous offsets rather than supervise prompt characters.
        if any(right > boundary for left, right in offsets[:prompt_count]) or any(left < boundary for left, right in offsets[prompt_count:]):
            raise ValueError(f'{r["row_id"]}: ambiguous offset ordering at boundary')
        boundary_method = 'offsets'
    cids = joined[prompt_count:]
    eos = tokenizer.eos_token_id
    if eos is None or not cids:
        raise ValueError(f'{r["row_id"]}: missing EOS or empty completion')
    special = set(getattr(tokenizer, 'all_special_ids', [eos]))
    if any(t in special for t in joined):
        raise ValueError(f'{r["row_id"]}: unexpected special tokens inside raw text; inspect with Person 1.')
    ids = joined + [eos]
    labels = [-100] * prompt_count + cids + [eos]
    if len(ids) > cap:
        raise ValueError(f'{r["row_id"]}: {len(ids)} tokens exceeds {cap}; no truncation is allowed.')
    counts = dict(prompt_tokens=prompt_count, completion_tokens=len(cids),
                  supervised_tokens=sum(x != -100 for x in labels[1:]), total_tokens=len(ids))
    if counts['supervised_tokens'] != len(cids) + 1:
        raise ValueError('Shifted target accounting failed')
    return dict(input_ids=ids, labels=labels, counts=counts, boundary_method=boundary_method)

def fingerprint(args):
    return dict(manifest_sha256=sha(args.data), template_sha256=sha(args.template),
                policy_sha256=sha(args.policy), model=MODEL, revision=REVISION,
                code_sha256=sha(__file__))

def csv_write(path, rows):
    if not rows:
        raise ValueError('No CSV records')
    with open(path, 'w', newline='', encoding='utf-8') as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0])); w.writeheader(); w.writerows(rows)

def reconcile(rows, tokenizer, template, policy):
    records, encoded = [], []
    for r in rows:
        rec = dict(row_id=r['row_id'], problem_group_id=r['problem_group_id'], dataset_id=r['dataset_id'])
        try:
            e = encode_row(r, tokenizer, template, policy['max_total_tokens'])
            for field in COUNT_FIELDS:
                rec['person1_' + field] = int(r[field])
                rec['person2_' + field] = e['counts'][field]
            rec['actual_supervised_tokens'] = e['counts']['supervised_tokens']
            rec['boundary_method'] = e['boundary_method']
            rec['match'] = all(rec['person1_' + f] == rec['person2_' + f] for f in COUNT_FIELDS)
            rec['error'] = '' if rec['match'] else 'Token-count mismatch'
            encoded.append(e)
        except ValueError as exc:
            for field in COUNT_FIELDS:
                rec['person1_' + field], rec['person2_' + field] = int(r[field]), ''
            rec['actual_supervised_tokens'], rec['boundary_method'] = '', 'failed'
            rec['match'], rec['error'] = False, str(exc)
            encoded.append(None)
        records.append(rec)
    return records, encoded

def get_tokenizer():
    from transformers import AutoTokenizer
    return AutoTokenizer.from_pretrained(MODEL, revision=REVISION, trust_remote_code=False, use_fast=True)

def verify(args):
    out = Path(args.out); out.mkdir(parents=True, exist_ok=False)
    rows = load_rows(args.data); check_unique(rows, args.allow_repeats)
    template, policy = load_contract(args.template, args.policy)
    tok = get_tokenizer()
    records, enc = reconcile(rows, tok, template, policy)
    csv_write(out / 'person2_token_verification.csv', records)
    passed = all(r['match'] for r in records)
    summary = fingerprint(args)
    summary.update(passed=passed, rows=len(rows), allow_repeats=args.allow_repeats,
                   unique_examples=len({r['row_id'] for r in rows}), mismatches=sum(not r['match'] for r in records),
                   template_utf8_sha256=hashlib.sha256(template.encode('utf-8')).hexdigest(),
                   actual_transformers_version=importlib.metadata.version('transformers'),
                   scope_note='Verification covers only this exact manifest, not the whole Person 1 working pool.')
    if passed:
        summary['processed_supervised_tokens_one_pass'] = sum(e['counts']['supervised_tokens'] for e in enc)
        summary['source_supervised_tokens'] = dict(collections.Counter())
        for r, e in zip(rows, enc):
            k = source(r)
            summary['source_supervised_tokens'][k] = summary['source_supervised_tokens'].get(k, 0) + e['counts']['supervised_tokens']
        csv_write(out / 'mask_preview.csv', [dict(position=i, token_id=t, token=tok.convert_ids_to_tokens(t),
                                                 label=enc[0]['labels'][i], supervised=enc[0]['labels'][i] != -100)
                                             for i, t in enumerate(enc[0]['input_ids'])])
    dump(out / 'verification.json', summary)
    print(json.dumps(summary, indent=2))
    if not passed:
        raise SystemExit('STOP: send the verification CSV to Person 1. Do not change counts to make the check pass.')


def check_nesting(args):
    previous_ids = []
    result = []
    for path in args.data:
        rows = load_rows(path); check_unique(rows)
        ids = [r['row_id'] for r in rows]
        if len(ids) < len(previous_ids) or ids[:len(previous_ids)] != previous_ids:
            raise ValueError(f'{path}: not an ordered prefix extension of previous manifest')
        counts = collections.Counter(source(r) for r in rows)
        if counts['orca'] != counts['numina']:
            raise ValueError(f'{path}: not balanced by examples')
        result.append(dict(path=str(path), rows=len(rows), source_counts=dict(counts), sha256=sha(path)))
        previous_ids = ids
    print(json.dumps(result, indent=2))

def subset(args):
    rows = load_rows(args.data); check_unique(rows)
    chosen = []
    for k in ('orca', 'numina'):
        part = [r for r in rows if source(r) == k][:16]
        if len(part) != 16:
            raise ValueError(f'Need at least 16 {k} examples')
        chosen.extend(part)
    random.Random(42).shuffle(chosen)
    with open(args.out, 'x', encoding='utf-8') as f:
        for r in chosen:
            f.write(json.dumps(r, ensure_ascii=False) + '\n')
    print(f'Wrote {len(chosen)} provisional debug examples; discard their trained adapter.')

def batches_for_epoch(n, epoch, seed, accumulation):
    order = list(range(n)); random.Random(seed + epoch).shuffle(order)
    return [order[i:i + accumulation] for i in range(0, n, accumulation)]

def provenance_guard(args, rows, current):
    verified = read_json(args.verification)
    if verified.get('passed') is not True:
        raise ValueError('Token verification did not pass')
    for k, v in current.items():
        if verified.get(k) != v:
            raise ValueError(f'Verification stale or for other input: {k}')
    if args.mode == 'debug':
        if len(rows) != 32:
            raise ValueError('Debug mode requires exactly 32 examples')
        return None
    if args.mode == 'preliminary':
        if args.epochs != 1 or args.parent:
            raise ValueError('Preliminary runs require one epoch independently from base')
        check_unique(rows)
        counts = collections.Counter(source(r) for r in rows)
        if counts['orca'] != counts['numina']:
            raise ValueError('Preliminary manifest must be 50/50 by examples')
        return None  # No claim of evaluation/protection clearance.
    if not args.clearance:
        raise ValueError('Real runs require --clearance: documented manual review and DEV+TEST overlap clearance')
    clearance = read_json(args.clearance)
    for k in ('manual_reviews_confirmed', 'dev_and_test_decontamination_confirmed'):
        if clearance.get(k) is not True:
            raise ValueError(f'Clearance incomplete: {k}')
    if not clearance.get('evidence'):
        raise ValueError('Record actual review/overlap evidence paths in clearance; a boolean alone is insufficient')
    if current['manifest_sha256'] not in clearance.get('approved_manifest_sha256', []):
        raise ValueError('This manifest is not approved in the clearance file')
    if args.mode == 'curve':
        if args.epochs != 1 or args.parent:
            raise ValueError('Curve runs must start from base and run exactly one epoch')
        check_unique(rows)
        c = collections.Counter(source(r) for r in rows)
        if c['orca'] != c['numina']:
            raise ValueError('Curve manifest must be 50/50 by examples')
    if args.mode == 'branch' and (not args.parent or args.epochs != 1):
        raise ValueError('Branches need a parent and exactly one pass through the exposure manifest')
    return clearance

def environment(torch):
    names = ('torch', 'transformers', 'peft', 'accelerate', 'bitsandbytes', 'safetensors', 'numpy')
    return dict(python=platform.python_version(), packages={p: importlib.metadata.version(p) for p in names},
                cuda=torch.version.cuda, gpu=torch.cuda.get_device_name(0), gpu_count_used=1,
                gpu_memory_bytes=torch.cuda.get_device_properties(0).total_memory)

def build_model(dtype, adapter=None, trainable=True):
    import torch
    from transformers import AutoModelForCausalLM, BitsAndBytesConfig
    from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training
    quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                              bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=dtype)
    base = AutoModelForCausalLM.from_pretrained(MODEL, revision=REVISION, trust_remote_code=False,
        quantization_config=quant, torch_dtype=dtype, device_map={'': 0}, attn_implementation='eager')
    base.config.use_cache = False
    # Apply the same preparation/casts for reload checks and training.
    base = prepare_model_for_kbit_training(base, use_gradient_checkpointing=trainable,
                                           gradient_checkpointing_kwargs={'use_reentrant': False})
    if adapter:
        model = PeftModel.from_pretrained(base, str(adapter), is_trainable=trainable)
    else:
        model = get_peft_model(base, LoraConfig(task_type='CAUSAL_LM', r=16, lora_alpha=32,
                                              lora_dropout=0.05, target_modules='all-linear', bias='none'))
    return model

def probe_logits(model, encoded, dtype):
    import torch
    model.eval()
    # Probe logits at the end of a short prefix; no gold scoring or benchmark use here.
    ids = torch.tensor([encoded['input_ids'][:min(32, len(encoded['input_ids']))]], device='cuda')
    with torch.no_grad(), torch.autocast('cuda', dtype=dtype):
        return model(input_ids=ids, attention_mask=torch.ones_like(ids)).logits[0, -1].float().cpu()

def save_checkpoint(path, model, optimizer, scaler, tok, state, config, encoded, dtype):
    import torch
    import numpy as np
    path = Path(path)
    temp = path.with_name(path.name + '.incomplete')
    temp.mkdir(parents=True, exist_ok=False)
    model.save_pretrained(temp / 'adapter', safe_serialization=True)
    tok.save_pretrained(temp / 'tokenizer')
    probe = probe_logits(model, encoded[0], dtype)
    torch.save(probe, temp / 'probe.pt')
    payload = dict(state=state, optimizer=optimizer.state_dict(), scaler=scaler.state_dict(),
                   python_rng=random.getstate(), numpy_rng=np.random.get_state(),
                   torch_rng=torch.get_rng_state(), cuda_rng=torch.cuda.get_rng_state_all())
    torch.save(payload, temp / 'training_state.pt')
    dump(temp / 'run_config.json', config)
    temp.rename(path)
    model.train()

def load_training_state(path):
    import torch
    # Only load checkpoints created by this script that YOU trust: pickle can execute code.
    return torch.load(Path(path) / 'training_state.pt', map_location='cpu', weights_only=False)

def parent_budget_check(args, rows, encoded, parent_config, parent_state):
    if parent_config['mode'] != 'curve' or parent_config['rows'] != 1000 or not parent_state['state']['complete']:
        raise ValueError('Pilot branch parent must be a completed 1,000-example curve run')
    seen = set(parent_state['state']['seen_groups'])
    if args.arm != 'REPEAT' and any(r['problem_group_id'] in seen for r in rows):
        raise ValueError('Fresh branch contains a parent-seen problem group')
    if args.arm == 'REPEAT' and any(r['problem_group_id'] not in seen for r in rows):
        raise ValueError('REPEAT contains an unseen problem group')
    target = parent_state['state']['source_tokens']
    actual = collections.Counter()
    for r, e in zip(rows, encoded):
        actual[source(r)] += e['counts']['supervised_tokens']
    for k in ('orca', 'numina'):
        if not target.get(k) or abs(actual[k] - target[k]) / target[k] > 0.01:
            raise ValueError(f'{k} branch budget differs by more than 1%; ask Person 4 to reconcile whole-example selection.')
    return dict(parent_processed_tokens=parent_state['state']['processed_tokens'],
                target_source_tokens=target, actual_source_tokens=dict(actual))

def train(args):
    import torch
    import numpy as np
    from transformers import set_seed
    if not torch.cuda.is_available() or int(os.environ.get('WORLD_SIZE', '1')) != 1:
        raise ValueError('This training implementation requires one CUDA/NVIDIA GPU; no CPU/MPS or distributed fallback.')
    if args.epochs < 1 or args.accumulation < 1 or args.save_every < 1 or args.lr <= 0:
        raise ValueError('epochs, accumulation, save-every and learning rate must be positive')
    out = Path(args.out)
    if out.exists():
        raise ValueError('Use a NEW output folder for every run/resume segment; existing output is never overwritten.')
    rows = load_rows(args.data)
    check_unique(rows, allow_repeats=args.mode == 'branch' and args.arm == 'REPEAT')
    template, policy = load_contract(args.template, args.policy)
    current = fingerprint(args)
    clearance = provenance_guard(args, rows, current)
    tok = get_tokenizer()
    records, encoded = reconcile(rows, tok, template, policy)
    if not all(r['match'] for r in records):
        raise ValueError('Actual tokenizer reconciliation failed immediately before training')
    dtype = torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8 and torch.cuda.is_bf16_supported() else torch.float16
    env = environment(torch)
    config = dict(**current, experiment_status='preliminary' if args.mode == 'preliminary' else args.mode,
                  evaluation_readiness='pending' if args.mode == 'preliminary' else 'see_mode_and_clearance',
                  mode=args.mode, arm=args.arm if args.mode == 'branch' else None,
                  rows=len(rows), epochs=args.epochs, seed=args.seed, accumulation=args.accumulation,
                  learning_rate=args.lr, warmup_updates=10, microbatch_size=1, packing=False,
                  dtype=str(dtype), environment=env, parent=str(Path(args.parent).resolve()) if args.parent else None,
                  branch_optimizer_policy='restore_parent_no_new_warmup' if args.mode == 'branch' else None,
                  clearance_sha256=sha(args.clearance) if clearance else None)
    restore = None
    parent_state = None
    adapter = None
    if args.parent:
        if args.mode != 'branch':
            raise ValueError('--parent is only for branch mode')
        pcfg = read_json(Path(args.parent) / 'run_config.json')
        for k in ('model', 'revision', 'template_sha256', 'policy_sha256', 'code_sha256', 'dtype', 'learning_rate', 'accumulation'):
            if pcfg.get(k) != config[k]:
                raise ValueError(f'Parent configuration differs: {k}')
        if pcfg['environment']['packages'] != env['packages']:
            raise ValueError('Parent package versions differ; reproduce the parent environment')
        parent_state = load_training_state(args.parent)
        config['branch_budget'] = parent_budget_check(args, rows, encoded, pcfg, parent_state)
        config['parent_state_sha256'] = sha(Path(args.parent) / 'training_state.pt')
        config['parent_adapter_sha256'] = sha(Path(args.parent) / 'adapter' / 'adapter_model.safetensors')
        adapter = Path(args.parent) / 'adapter'
    if args.resume:
        previous = read_json(Path(args.resume) / 'run_config.json')
        if previous != config:
            raise ValueError('Resume config differs. Keep all inputs/seed/epochs/parent/environment identical.')
        restore = load_training_state(args.resume)
        adapter = Path(args.resume) / 'adapter'
    set_seed(args.seed)
    model = build_model(dtype, adapter=adapter)
    trainable = [(n, p) for n, p in model.named_parameters() if p.requires_grad]
    if not trainable or any('lora_' not in n for n, p in trainable):
        raise ValueError('Expected only LoRA parameters to be trainable')
    params = [p for n, p in trainable]
    optimizer = torch.optim.AdamW(params, lr=args.lr, weight_decay=0.0)
    scaler = torch.amp.GradScaler('cuda', enabled=dtype == torch.float16)
    state = dict(next_epoch=0, next_group=0, optimizer_updates=0, processed_tokens=0, input_tokens=0,
                 source_tokens={}, seen_groups=[], seen_ids=[], training_seconds=0.0, complete=False,
                 parent_optimizer_updates=0)
    if parent_state:
        optimizer.load_state_dict(parent_state['optimizer']); scaler.load_state_dict(parent_state['scaler'])
        state['parent_optimizer_updates'] = parent_state['state']['optimizer_updates']
        # Same branch seed + fresh deterministic data order; parent optimizer restored for every arm.
    if restore:
        state = restore['state']
        if state['complete']:
            raise ValueError('Checkpoint already completed; use branch mode for new data')
        optimizer.load_state_dict(restore['optimizer']); scaler.load_state_dict(restore['scaler'])
    # Prevent accidental same-object/no-update success in the debug validation.
    initial = {n: p.detach().float().cpu().clone() for n, p in trainable} if args.mode == 'debug' else None
    before = probe_logits(model, encoded[0], dtype)
    if args.resume:
        saved_probe = torch.load(Path(args.resume) / 'probe.pt', weights_only=True)
        if not torch.allclose(saved_probe, before, rtol=1e-3, atol=1e-3):
            raise ValueError('Reload logit probe differs; inspect loading/precision before resuming')
    # Set RNG after model construction/probe. On resume restore exact checkpoint RNG.
    if restore:
        random.setstate(restore['python_rng']); np.random.set_state(restore['numpy_rng'])
        torch.set_rng_state(restore['torch_rng']); torch.cuda.set_rng_state_all(restore['cuda_rng'])
    else:
        set_seed(args.seed)
    out.mkdir(parents=True)
    dump(out / 'run_config.json', config)
    dump(out / 'trainable_parameters.json', {'names': [n for n, p in trainable], 'count': sum(p.numel() for p in params)})
    (out / 'requirements.lock.txt').write_text(subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True))
    model.train(); torch.cuda.reset_peak_memory_stats()
    start = time.perf_counter(); old_seconds = state['training_seconds']
    seen_ids, seen_groups = set(state['seen_ids']), set(state['seen_groups'])
    segment_updates = 0
    stop = False
    with open(out / 'training_log.jsonl', 'w', encoding='utf-8') as log:
        for epoch in range(state['next_epoch'], args.epochs):
            groups = batches_for_epoch(len(rows), epoch, args.seed, args.accumulation)
            first = state['next_group'] if epoch == state['next_epoch'] else 0
            for g in range(first, len(groups)):
                indices = groups[g]
                target_total = sum(encoded[i]['counts']['supervised_tokens'] for i in indices)
                update = state['optimizer_updates'] + 1
                lr = args.lr if args.mode == 'branch' else args.lr * min(update / 10.0, 1.0)
                for pg in optimizer.param_groups:
                    pg['lr'] = lr
                optimizer.zero_grad(set_to_none=True)
                weighted_loss = 0.0
                for i in indices:
                    e = encoded[i]
                    ids = torch.tensor([e['input_ids']], dtype=torch.long, device='cuda')
                    labels = torch.tensor([e['labels']], dtype=torch.long, device='cuda')
                    with torch.autocast('cuda', dtype=dtype):
                        logits = model(input_ids=ids, attention_mask=torch.ones_like(ids)).logits
                        # Explicit next-token objective. Chunk float32 cross entropy to limit
                        # temporary memory for Qwen's large vocabulary. No label shift twice.
                        loss_sum = torch.zeros((), device='cuda')
                        for pos in range(0, ids.shape[1] - 1, 128):
                            end = min(pos + 128, ids.shape[1] - 1)
                            loss_sum = loss_sum + torch.nn.functional.cross_entropy(
                                logits[0, pos:end].float(), labels[0, pos + 1:end + 1],
                                ignore_index=-100, reduction='sum')
                        loss = loss_sum / target_total
                    if not torch.isfinite(loss):
                        raise RuntimeError('Nonfinite loss; stop and diagnose data/precision before comparing runs')
                    weighted_loss += loss.detach().item()
                    scaler.scale(loss).backward()
                    del logits, loss_sum, loss, ids, labels
                scaler.unscale_(optimizer)
                grad_norm = torch.nn.utils.clip_grad_norm_(params, 1.0)
                if not torch.isfinite(grad_norm):
                    raise RuntimeError('Nonfinite gradients: no skipped updates are silently counted; diagnose precision and restart from checkpoint.')
                scaler.step(optimizer); scaler.update()
                state['optimizer_updates'] += 1; segment_updates += 1
                state['processed_tokens'] += target_total
                state['input_tokens'] += sum(encoded[i]['counts']['total_tokens'] for i in indices)
                for i in indices:
                    r, e = rows[i], encoded[i]
                    seen_ids.add(r['row_id']); seen_groups.add(r['problem_group_id'])
                    k = source(r)
                    state['source_tokens'][k] = state['source_tokens'].get(k, 0) + e['counts']['supervised_tokens']
                state['seen_ids'], state['seen_groups'] = sorted(seen_ids), sorted(seen_groups)
                state['next_epoch'], state['next_group'] = (epoch + 1, 0) if g + 1 == len(groups) else (epoch, g + 1)
                torch.cuda.synchronize()
                state['training_seconds'] = old_seconds + time.perf_counter() - start
                record = dict(epoch=epoch, optimizer_update=state['optimizer_updates'], target_tokens=target_total,
                              cumulative_processed_tokens=state['processed_tokens'], loss=weighted_loss,
                              lr=lr, grad_norm=float(grad_norm), elapsed_seconds=state['training_seconds'],
                              peak_allocated_GB=torch.cuda.max_memory_allocated() / 1e9)
                log.write(json.dumps(record) + '\n'); log.flush()
                print(json.dumps(record), flush=True)
                if args.stop_after and segment_updates >= args.stop_after:
                    stop = True; break
                if state['optimizer_updates'] % args.save_every == 0:
                    save_checkpoint(out / f'checkpoint-{state["optimizer_updates"]:06d}', model, optimizer, scaler,
                                    tok, state, config, encoded, dtype)
            if stop:
                break
    state['complete'] = state['next_epoch'] >= args.epochs
    final_path = out / ('final' if state['complete'] else 'paused')
    save_checkpoint(final_path, model, optimizer, scaler, tok, state, config, encoded, dtype)
    csv_write(out / 'seen_groups.csv', [dict(problem_group_id=g) for g in sorted(seen_groups)])
    if args.mode == 'curve' and len(rows) == 1000 and state['complete']:
        csv_write(out / 'parent_seen_groups_R1000.csv', [dict(problem_group_id=g) for g in sorted(seen_groups)])
    actual_unique = {r['row_id']: e['counts']['supervised_tokens'] for r, e in zip(rows, encoded) if r['row_id'] in seen_ids}
    summary = dict(**state, experiment_status=config['experiment_status'],
                   evaluation_readiness=config['evaluation_readiness'], unique_examples=len(seen_ids), unique_supervised_tokens=sum(actual_unique.values()),
                   peak_allocated_GB=torch.cuda.max_memory_allocated() / 1e9,
                   seconds_include_checkpoint_overhead=True, checkpoint=str(final_path.resolve()))
    if parent_state:
        summary['total_parent_plus_branch_tokens'] = parent_state['state']['processed_tokens'] + state['processed_tokens']
    if initial is not None:
        delta = max((p.detach().float().cpu() - initial[n]).abs().max().item() for n, p in trainable)
        summary['max_adapter_delta_this_segment'] = delta
        if not delta > 0:
            raise RuntimeError('Debug adapters did not change')
    dump(out / 'training_summary.json', summary)
    print(f'Saved {final_path}. Evaluate with your separately validated Person 3 evaluator.')

def reload_check(args):
    import torch
    cfg = read_json(Path(args.checkpoint) / 'run_config.json')
    if cfg['model'] != MODEL or cfg['revision'] != REVISION:
        raise ValueError('Checkpoint model identity differs')
    for k in ('manifest_sha256', 'template_sha256', 'policy_sha256', 'code_sha256', 'wrapper_sha256', 'reference_code_sha256'):
        if fingerprint(args)[k] != cfg[k]:
            raise ValueError('Reload check must use original data/template/policy/code')
    dtype = torch.bfloat16 if cfg['dtype'] == 'torch.bfloat16' else torch.float16
    rows = load_rows(args.data); template, policy = load_contract(args.template, args.policy)
    enc = encode_row(rows[0], get_tokenizer(), template, policy['max_total_tokens'])
    model = build_model(dtype, Path(args.checkpoint) / 'adapter', trainable=False)
    actual = probe_logits(model, enc, dtype)
    expected = torch.load(Path(args.checkpoint) / 'probe.pt', weights_only=True)
    maximum = (actual - expected).abs().max().item()
    passed = torch.allclose(actual, expected, rtol=1e-3, atol=1e-3)
    print(json.dumps(dict(passed=passed, max_absolute_logit_difference=maximum)))
    if not passed:
        raise SystemExit('Reload mismatch: inspect before real runs')

def main():
    p = argparse.ArgumentParser(description=__doc__)
    sub = p.add_subparsers(dest='command', required=True)
    def common(q):
        q.add_argument('--data', required=True); q.add_argument('--template', required=True); q.add_argument('--policy', required=True)
    q = sub.add_parser('check-nesting'); q.add_argument('--data', nargs='+', required=True); q.set_defaults(func=check_nesting)
    q = sub.add_parser('subset'); q.add_argument('--data', required=True); q.add_argument('--out', required=True); q.set_defaults(func=subset)
    q = sub.add_parser('verify'); common(q); q.add_argument('--out', required=True); q.add_argument('--allow-repeats', action='store_true'); q.set_defaults(func=verify)
    q = sub.add_parser('train'); common(q)
    q.add_argument('--verification', required=True); q.add_argument('--out', required=True)
    q.add_argument('--mode', choices=['debug', 'preliminary', 'curve', 'branch'], required=True)
    q.add_argument('--clearance'); q.add_argument('--epochs', type=int, default=1)
    q.add_argument('--seed', type=int, default=42); q.add_argument('--accumulation', type=int, default=8)
    q.add_argument('--lr', type=float, default=1e-4); q.add_argument('--save-every', type=int, default=100)
    q.add_argument('--parent'); q.add_argument('--resume'); q.add_argument('--stop-after', type=int)
    q.add_argument('--arm', choices=['REPEAT', 'RANDOM', 'FIT', 'MATCHED_RANDOM'])
    q.set_defaults(func=train)
    q = sub.add_parser('reload-check'); common(q); q.add_argument('--checkpoint', required=True); q.set_defaults(func=reload_check)
    args = p.parse_args()
    if args.command == 'train' and args.mode == 'branch' and not args.arm:
        p.error('Branch mode requires --arm')
    if args.command == 'train' and args.mode != 'branch' and args.arm:
        p.error('--arm only belongs to branch mode')
    args.func(args)

if __name__ == '__main__':
    main()


## 3. Write the two-model data/integration wrapper
This selects the correct pinned model, checks release hashes, uses Person 1’s encoder, checks exact tokenizer compatibility for 1.5B, exports actual-label evidence, and enforces readiness. The model argument and separate output folders prevent one model’s results from replacing the other’s.

In [ ]:
%%writefile person2.py
"""Person 2 v3 integration; run --help. NVIDIA GPU needed only for training/reload."""
import argparse, csv, hashlib, json, sys
from pathlib import Path
from datetime import datetime, timezone
from types import SimpleNamespace
import trainer_core as core
from capstone_person1 import person1_data_contract as contract
from functools import lru_cache

ROOT = Path(__file__).resolve().parent
VERSION = 'person2-dual-v3-20261015'
REFERENCE_ROOT = Path(contract.__file__).resolve().parent
REFERENCE_FILES = ['__init__.py','person1_data_contract.py','pipeline_controls.py','instance_matching.py']
MODELS = {
    '0.5B': ('Qwen/Qwen2.5-0.5B', '060db6499f32faf8b98477b0a26969ef7d8b9987'),
    '1.5B': ('Qwen/Qwen2.5-1.5B', '8faed761d45a263340a0528343f099c05c9a4323'),
}
SELECTED_MODEL = '0.5B'

@lru_cache(maxsize=2)
def model_tokenizer(model_id, revision):
    from transformers import AutoTokenizer
    tok = AutoTokenizer.from_pretrained(model_id, revision=revision, use_fast=True, trust_remote_code=False)
    contract.verify_tokenizer(tok)
    return tok

def get_tokenizer():
    return model_tokenizer(core.MODEL,core.REVISION)

def select_model(name):
    global SELECTED_MODEL
    SELECTED_MODEL = name
    core.MODEL, core.REVISION = MODELS[name]

def now(): return datetime.now(timezone.utc).isoformat()
def read(p): return json.loads(Path(p).read_text())
def write(p, obj): core.dump(p, obj)
def digest(p): return core.sha(p)

def collate(encoded):
    # Used both in verification and the actual training encoder.
    row = contract.pad_encoded(encoded, len(encoded['input_ids']))
    n, p = len(row['input_ids']), encoded['prompt_tokens']
    assert row['labels'][:p] == [-100]*p
    assert row['attention_mask'] == [1]*n
    assert row['labels'][-1] == contract.EXPECTED_END_ID
    assert sum(v != -100 for v in row['labels'][1:]) == encoded['supervised_tokens']
    return row

def encode(r, tokenizer, template, cap):
    if template != contract.PROMPT_TEMPLATE: raise ValueError('Prompt contract differs')
    e = contract.validate_manifest_record(r, tokenizer, max_total_tokens=cap)
    row = collate(e)
    if SELECTED_MODEL == '1.5B':
        reference_tok = model_tokenizer(*MODELS['0.5B'])
        reference = contract.validate_manifest_record(r, reference_tok, max_total_tokens=cap)
        # Equal counts alone do not prove equal tokenization. Compare actual IDs and masks too.
        for key in ('input_ids','labels','attention_mask'):
            if row[key] != reference[key]:
                raise ValueError(f'1.5B tokenizer differs on {r["row_id"]}: {key}. Reconcile budgets with Person 1; no silent reuse.')
    # Actual microbatch size is 1; no padding is needed in the training path.
    # Exercise the same positional-padding helper with extra padding in verification.
    padded = contract.pad_encoded(e, len(e['input_ids'])+3)
    assert padded['labels'][-3:] == [-100]*3
    assert padded['attention_mask'][-3:] == [0]*3
    assert padded['labels'][len(e['input_ids'])-1] == contract.EXPECTED_END_ID
    assert sum(x != -100 for x in padded['labels']) == e['supervised_tokens']
    return dict(input_ids=row['input_ids'], labels=row['labels'],
                counts={k:e[k] for k in core.COUNT_FIELDS}, boundary_method='v3_combined_offsets')

_original_fingerprint = core.fingerprint

def fingerprint(args):
    f = _original_fingerprint(args)
    f['wrapper_sha256'] = digest(__file__)
    f['reference_code_sha256'] = {p.name:digest(p) for p in [REFERENCE_ROOT/name for name in REFERENCE_FILES]}
    return f

core.encode_row = encode
core.fingerprint = fingerprint
core.get_tokenizer = get_tokenizer


def setup(release, scope, work):
    release, work = Path(release).resolve(), Path(work).resolve()
    if work == release or release in work.parents: raise ValueError('WORK must be outside frozen release')
    meta = read(release/'metadata/release_manifest.json')
    if not meta['readiness']['data_ready_for_trainer']: raise ValueError('Release is not data-ready')
    for relative in [f'manifests/{scope}.jsonl', f'manifests/{scope}_summary.json', 'configs/formatting_contract.json']:
        path = release/relative
        if not path.exists(): raise FileNotFoundError(f'Missing {path}. Download archive 01; release_records is metadata only.')
        if digest(path) != meta['files'][relative]['sha256']: raise ValueError(f'Release hash mismatch: {relative}')
    origin_path = release/'metadata/encoder_origin_manifest.json'
    if digest(origin_path) != meta['original_manifest_sha256']:
        raise ValueError('Encoder origin manifest hash mismatch')
    origin = read(origin_path)
    for path in [REFERENCE_ROOT/name for name in REFERENCE_FILES]:
        key = 'code/capstone_person1/'+path.name
        expected = meta['files'].get(key, origin['files'].get(key, {})).get('sha256')
        if digest(path) != expected: raise ValueError(f'Encoder differs from verified provenance: {key}')
    cfg = read(release/'configs/formatting_contract.json')
    if cfg['contract_version'] != contract.CONTRACT_VERSION: raise ValueError('Unsupported contract')
    work.mkdir(parents=True, exist_ok=True)
    template, policy = work/'prompt_template.json', work/'tokenization_policy.json'
    write(template, {'template': cfg['prompt_template']})
    write(policy, dict(confirmed_with_person1=True, add_special_tokens=False, append_eos=True,
        max_total_tokens=cfg['max_total_tokens'], provenance=f"{release}; {contract.CONTRACT_VERSION}"))
    return SimpleNamespace(data=str(release/f'manifests/{scope}.jsonl'), template=str(template), policy=str(policy),
        release=release, scope=scope, work=work, allow_repeats=False)


def verify(a, who):
    a.out = a.work/'checks'/a.scope
    # Re-encode and validate aggregate counts against the actual release summary.
    summary = read(a.release/f'manifests/{a.scope}_summary.json')
    contract.validate_manifest(a.data, summary, tokenizer=get_tokenizer())
    core.verify(a)
    rows = list(csv.DictReader((a.out/'person2_token_verification.csv').open()))
    with (a.out/'actual_labels.csv').open('w', newline='') as f:
        w=csv.DictWriter(f,fieldnames=['row_id','actual_supervised_tokens']);w.writeheader()
        w.writerows({k:r[k] for k in w.fieldnames} for r in rows)
    write(a.out/'actual_labels_metadata.json', dict(trainer_version=VERSION,
        contract_version=contract.CONTRACT_VERSION, model=core.MODEL, model_revision=core.REVISION,
        tokenizer_compatibility='every row IDs/labels/attention matched 0.5B reference' if SELECTED_MODEL=='1.5B' else 'reference model', manifest_sha256=digest(a.data),
        prompt_mask_verified=True, eos_label_verified=True, padding_by_position_verified=True,
        confirmed_by=who, confirmed_utc=now(), check_method='programmatic assertions on actual collator output; inspect mask_preview.csv',
        padding_note='training microbatch=1, no padding; extra-padding regression executed for every row'))
    print('Send actual_labels.csv and actual_labels_metadata.json to Person 1; inspect mask_preview.csv.')


def clearance(a, path):
    c = read(path)
    # Never turn a DATA_READY status into evaluation clearance automatically.
    for key in ('person3_evaluator_frozen','development_and_test_protection_verified'):
        if c.get(key) is not True: raise ValueError('Team readiness incomplete: '+key)
    if c.get('release_manifest_sha256') != digest(a.release/'metadata/release_manifest.json'):
        raise ValueError('Readiness is for a different data release')
    if not c.get('confirmed_by') or not c.get('confirmed_utc'): raise ValueError('Actual identity/time required')
    for key in ('evaluation_contract','overlap_report'):
        evidence=c.get(key,{})
        p=Path(evidence.get('path',''))
        if not p.is_file() or digest(p)!=evidence.get('sha256'): raise ValueError('Missing/stale evidence: '+key)
    return dict(manual_reviews_confirmed=True,dev_and_test_decontamination_confirmed=True,
        evidence={'team_readiness_path':str(Path(path).resolve()),'team_readiness_sha256':digest(path)},
        approved_manifest_sha256=[digest(a.data)])


def train(a, ns):
    debug=a.scope=='debug_32'
    a.mode='debug' if debug else ('preliminary' if ns.preliminary else 'curve')
    a.epochs=ns.debug_epochs if debug else 1
    a.seed=ns.seed; a.accumulation=8;a.lr=1e-4;a.save_every=100
    a.parent=None;a.resume=ns.resume;a.stop_after=ns.stop_after;a.arm=None
    a.verification=a.work/'checks'/a.scope/'verification.json'
    a.out=a.work/'runs'/ns.run_name;a.clearance=None
    if not debug:
        if ns.preliminary and ns.readiness:
            raise ValueError('Choose preliminary or readiness-backed training, not both')
        if not ns.preliminary and not ns.readiness:
            raise ValueError('Use --preliminary for a pending-evaluation experiment, or supply --readiness')
        gate=None if ns.preliminary else clearance(a,ns.readiness)
        # Require actual debug training AND reload evidence before main runs.
        report=read(a.work/'debug_reload_passed.json')
        if report['release_manifest_sha256']!=digest(a.release/'metadata/release_manifest.json'):
            raise ValueError('Debug was for a different release')
        if report['fingerprint'] != fingerprint(setup(a.release,'debug_32',a.work)):
            raise ValueError('Debug code/config has changed')
        if digest(Path(report['checkpoint'])/'adapter/adapter_model.safetensors')!=report['adapter_sha256']:
            raise ValueError('Debug adapter changed')
        if gate is not None:
            a.clearance=a.work/(ns.run_name+'_clearance.json');write(a.clearance,gate)
        # Selected manifest is verified by core.train. No R1000 dependency for preliminary runs.
        if not ns.preliminary:
            v=read(a.work/'checks/train_n1000/verification.json')
            ref=fingerprint(setup(a.release,'train_n1000',a.work))
            if not v.get('passed') or any(v.get(k)!=value for k,value in ref.items()):
                raise ValueError('Verify all R1000 labels first for a formal curve run')
    core.train(a)
    if a.mode == 'preliminary':
        write(a.out/'PRELIMINARY_STATUS.json', dict(
            experiment_status='preliminary', person3_evaluator_frozen=False,
            development_and_test_protection_verified=False,
            note='These checks have not been established by this run; false does not mean overlap was found.',
            release_manifest_sha256=digest(a.release/'metadata/release_manifest.json'),
            created_utc=now(), intended_use='Checkpoint loading and preliminary evaluation; final study eligibility pending'))


def reload(a, ns):
    a.checkpoint=Path(ns.checkpoint).resolve()
    core.reload_check(a)
    summary = read(a.checkpoint.parent/'training_summary.json')
    if not summary.get('complete'):
        raise ValueError('A paused checkpoint reload does not establish a completed smoke/main run')
    report=dict(status='passed',checkpoint=str(a.checkpoint),adapter_sha256=digest(a.checkpoint/'adapter/adapter_model.safetensors'),
        release_manifest_sha256=digest(a.release/'metadata/release_manifest.json'),fingerprint=fingerprint(a),verified_utc=now())
    write(a.work/('debug_reload_passed.json' if a.scope=='debug_32' else a.scope+'_reload_passed.json'),report)


def parent(a, ns):
    if a.scope!='train_n1000': raise ValueError('Parent binding requires train_n1000')
    run=Path(ns.run).resolve();s=read(run/'training_summary.json');cfg=read(run/'run_config.json')
    if cfg.get('mode') != 'curve': raise ValueError('Only readiness-backed curve runs can become formal parents')
    if not s['complete'] or s['unique_examples']!=1000 or cfg['manifest_sha256']!=digest(a.data):
        raise ValueError('Not a completed matching R1000')
    check=read(a.work/'train_n1000_reload_passed.json')
    adapter=run/'final/adapter/adapter_model.safetensors'
    if check['adapter_sha256']!=digest(adapter) or check['checkpoint']!=str(run/'final'): raise ValueError('Reload this checkpoint first')
    # Match observed seen groups exactly and check selection freshness.
    groups={r['problem_group_id'] for r in core.load_rows(a.data)}
    actual={r['problem_group_id'] for r in csv.DictReader((run/'seen_groups.csv').open())}
    if actual!=groups: raise ValueError('Actual seen groups differ')
    selection=a.release/'manifests/selection_bank.jsonl'
    meta=read(a.release/'metadata/release_manifest.json')
    if digest(selection)!=meta['files']['manifests/selection_bank.jsonl']['sha256']: raise ValueError('Selection hash differs')
    if groups & {r['problem_group_id'] for r in core.load_rows(selection)}: raise ValueError('Selection overlap')
    write(run/'actual_parent.json',dict(actual_run_id=run.name,checkpoint_id=str(run/'final'),
        checkpoint_sha256=digest(adapter),checkpoint_digest_scope='final/adapter/adapter_model.safetensors',
        actual_manifest_sha256=digest(a.data),confirmed_by=ns.confirmed_by,confirmed_utc=now()))
    print('Parent evidence ready:',run/'actual_parent.json')


def main():
    p=argparse.ArgumentParser(description=__doc__)
    p.add_argument('command',choices=['verify','train','reload','parent'])
    p.add_argument('--model',choices=list(MODELS),required=True)
    p.add_argument('--release',required=True);p.add_argument('--work',required=True)
    p.add_argument('--scope',choices=['debug_32']+[f'train_n{n}' for n in [500,1000,2000,4000,8000,16000,40000]],required=True)
    p.add_argument('--confirmed-by');p.add_argument('--run-name');p.add_argument('--readiness')
    p.add_argument('--preliminary',action='store_true',help='Train with evaluation/protection readiness explicitly pending')
    p.add_argument('--debug-epochs',type=int,default=10);p.add_argument('--seed',type=int,default=42)
    p.add_argument('--checkpoint');p.add_argument('--run');p.add_argument('--resume');p.add_argument('--stop-after',type=int)
    n=p.parse_args()
    if n.command in ['verify','parent'] and not n.confirmed_by:p.error('--confirmed-by required')
    if n.command=='train' and not n.run_name:p.error('--run-name required')
    if n.command=='reload' and not n.checkpoint:p.error('--checkpoint required')
    if n.command=='parent' and not n.run:p.error('--run required')
    select_model(n.model)
    a=setup(n.release,n.scope,n.work)
    {'verify':lambda:verify(a,n.confirmed_by),'train':lambda:train(a,n),
     'reload':lambda:reload(a,n),'parent':lambda:parent(a,n)}[n.command]()

if __name__=='__main__':main()


## 4. Write dependency requirements
Uses the same pinned dependencies as the previous notebook.


In [ ]:
%%writefile download_data.py
"""Download only archive 01, verify the supplied SHA256, safely extract it."""
import argparse, hashlib, shutil, urllib.request, zipfile
from pathlib import Path
NAME='person1-v3-01-training-and-handoff.zip'
URL='https://github.com/jahnvisethjs/infj-llm-distillation-improvement/releases/download/person1-data-v3-20261012/'+NAME
SHA='a4cbe6a494260ef3205a6391da67b28d3f7eb6bfe3fabf0df6e8708b077bca86'
p=argparse.ArgumentParser(description=__doc__);p.add_argument('--destination',required=True);p.add_argument('--archive',help='Use an already downloaded archive instead')
a=p.parse_args();root=Path(a.destination).resolve();root.mkdir(parents=True,exist_ok=True)
archive=Path(a.archive).resolve() if a.archive else root/NAME
if not archive.exists():
 if a.archive:raise FileNotFoundError(archive)
 print('Downloading approximately 302 MiB from the URL documented by Person 1...')
 temp=archive.with_suffix('.partial')
 try:
  with urllib.request.urlopen(URL,timeout=60) as response, temp.open('wb') as f:shutil.copyfileobj(response,f)
  temp.rename(archive)
 except Exception:
  temp.unlink(missing_ok=True);raise
h=hashlib.sha256()
with archive.open('rb') as f:
 for block in iter(lambda:f.read(1024*1024),b''):h.update(block)
if h.hexdigest()!=SHA:raise ValueError('Archive checksum mismatch. Do not use these bytes.')
with zipfile.ZipFile(archive) as z:
 for member in z.infolist():
  target=(root/member.filename).resolve()
  if root not in target.parents or (member.external_attr >> 16)&0o170000==0o120000:raise ValueError('Unsafe ZIP member')
  if target.exists() and target.is_file():
   with z.open(member) as src:
    if hashlib.sha256(src.read()).hexdigest()!=hashlib.sha256(target.read_bytes()).hexdigest():
     raise ValueError('Refusing to overwrite different existing file: '+str(target))
 z.extractall(root)
print('Data extracted:',root/'capstone_person1/releases/person1_v3_20261012_ready')


In [ ]:
%%writefile requirements.txt
# Proposed reproducible starting stack, NOT GPU-validated in the authoring environment.
# First install/use a CUDA-compatible PyTorch >=2.6,<3 for your actual NVIDIA driver.
# In Colab, inspect the supplied torch before replacing it. Restart after installation.
transformers==4.57.1
peft==0.17.1
accelerate==1.10.1
bitsandbytes==0.48.1
safetensors>=0.4,<1
numpy>=1.26,<3
pandas>=2.2,<3
pyarrow>=17,<23


## 5. Install the isolated Python environment
This may download several GB on first setup. No change to SOL’s shared environment. The environment is reused on later sessions. Actual GPU compatibility is tested below.

In [ ]:
if not PYTHON.exists():
    subprocess.run([str(TRAINING_PYTHON),'-m','venv',str(ENV)],check=True)
# Check even if the environment already exists; do not install into a wrong-version venv.
subprocess.run([
    str(PYTHON), '-c',
    "import sys; print('Environment Python:', sys.version); "
    "assert sys.version_info[:2] == (3, 12), 'Existing environment uses a different Python. Set ENV to a new folder in cell 1 and rerun setup.'"
], check=True)
marker = ENV/'person2_install_complete.json' 
if not marker.exists():
    subprocess.run([str(PYTHON),'-m','pip','install','--upgrade','pip'],check=True)
    subprocess.run([str(PYTHON),'-m','pip','install','torch==2.6.0',
                    '--index-url','https://download.pytorch.org/whl/cu124'],check=True)
    subprocess.run([str(PYTHON),'-m','pip','install','-r',str(PACKAGE/'requirements.txt')],check=True)
    subprocess.run([str(PYTHON),'-m','pip','check'],check=True)
    marker.write_text(json.dumps({'torch':'2.6.0','cuda_wheel':'cu124'}))
probe = """import torch, transformers, peft, bitsandbytes
print('Torch:',torch.__version__,'CUDA:',torch.version.cuda)
assert torch.cuda.is_available(), 'CUDA unavailable: check allocation/driver/environment'
assert torch.cuda.device_count()==1, 'Request exactly one GPU'
print('GPU:',torch.cuda.get_device_name(0))
x=torch.ones(16,device='cuda'); print('CUDA computation:',x.sum().item())
"""
subprocess.run([str(PYTHON),'-c',probe],check=True)


## 6. Validate the v4 release and assemble local inputs
Uses uploaded data and summaries. Reconstructs the debug summary exactly from the new manifest, verifies existing debug data against the new hash, and verifies encoder code against the original release identified by v4. Neither the old R1000 dataset nor its summary is required. No declared hashes are changed.


In [ ]:
OLD_RELEASE = DATA_ROOT/'capstone_person1/releases/person1_v3_20261012_ready'

def file_sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

new_manifest = UPLOADS/'release_manifest.json'
meta = json.loads(new_manifest.read_text())
origin_path = OLD_RELEASE/'metadata/release_manifest.json'
if file_sha256(origin_path) != meta.get('original_manifest_sha256'):
    raise RuntimeError('Original release manifest does not match v4 encoder provenance.')
origin = json.loads(origin_path.read_text())
sources = {'configs/formatting_contract.json': UPLOADS/'formatting_contract.json'}
for size in DATASET_SIZES:
    for filename in [f'train_n{size}.jsonl', f'train_n{size}_summary.json']:
        sources[f'manifests/{filename}'] = UPLOADS/filename

# Existing debug data is reused only after checking its NEW release hash.
relative = 'manifests/debug_32.jsonl'
candidates = [UPLOADS/relative, UPLOADS/'debug_32.jsonl', OLD_RELEASE/relative]
sources[relative] = next((p for p in candidates if p.is_file()), candidates[-1])
# Reconstruct the exact new summary from the supplied manifest, not the stale v3 summary.
debug_summary = (json.dumps(meta['manifests']['debug_32'], indent=2, ensure_ascii=False)+'\n').encode()
if hashlib.sha256(debug_summary).hexdigest() != meta['files']['manifests/debug_32_summary.json']['sha256']:
    raise RuntimeError('Embedded debug summary cannot reproduce its declared hash.')

errors = []
for relative, source in sources.items():
    expected = meta.get('files', {}).get(relative, {}).get('sha256')
    if not source.is_file(): errors.append(f'Missing source: {source}')
    elif not expected: errors.append(f'New manifest has no hash for: {relative}')
    elif file_sha256(source) != expected: errors.append(f'File does not match new manifest: {relative}')

# The v4 manifest points to the original release but does not republish its encoder hashes.
for filename in ['__init__.py', 'person1_data_contract.py', 'pipeline_controls.py', 'instance_matching.py']:
    relative = f'code/capstone_person1/{filename}'
    source = OLD_RELEASE/relative
    expected = meta['files'].get(relative, origin['files'].get(relative, {})).get('sha256')
    if not source.is_file(): errors.append(f'Missing encoder: {source}')
    elif not expected or file_sha256(source) != expected: errors.append(f'Encoder provenance mismatch: {relative}')
    sources[relative] = source
if errors:
    raise RuntimeError('Corrected release is not ready:\n'+'\n'.join(errors))

# Confirm seed-only comparisons use the exact data from a completed seed-42 run.
comparison = []
for model in MODEL_SIZES:
    model_folder = 'qwen2_5_' + model.lower().replace('.', '_')
    for size in DATASET_SIZES:
        actual_hash = file_sha256(UPLOADS/f'train_n{size}.jsonl')
        matches = []
        for root in SEED42_ROOTS:
            for cfg_path in (root/model_folder/'runs').glob(f'R{size}_seed42*/run_config.json'):
                cfg = json.loads(cfg_path.read_text())
                summary_path = cfg_path.parent/'training_summary.json'
                if (cfg.get('seed') == 42 and cfg.get('rows') == size
                    and cfg.get('manifest_sha256') == actual_hash
                    and summary_path.is_file()
                    and json.loads(summary_path.read_text()).get('complete') is True):
                    matches.append(str(cfg_path))
        if not matches:
            raise RuntimeError(f'No completed seed-42 run with matching data for {model}, n={size}. '
                'If stored elsewhere, add its RESULTS root to SEED42_ROOTS in Settings. '
                'If the uploaded dataset changed, reconcile it before calling this a seed-only comparison.')
        comparison.append(dict(model=model, examples=size, manifest_sha256=actual_hash,
                               matching_seed42_configs=matches))
(WORK/'seed42_data_comparison.json').write_text(json.dumps(comparison, indent=2))

copies = {**sources, 'metadata/release_manifest.json': new_manifest,
          'metadata/encoder_origin_manifest.json': origin_path}
# Validate existing destination files before writing.
for relative, source in copies.items():
    destination = RELEASE/relative
    if destination.exists() and file_sha256(destination) != file_sha256(source):
        raise RuntimeError(f'Different file already exists: {destination}; use a new RELEASE folder.')
debug_path = RELEASE/'manifests/debug_32_summary.json'
if debug_path.exists() and debug_path.read_bytes() != debug_summary:
    raise RuntimeError(f'Different debug summary already exists: {debug_path}')
for relative, source in copies.items():
    destination = RELEASE/relative
    destination.parent.mkdir(parents=True, exist_ok=True)
    if not destination.exists(): shutil.copy2(source, destination)
debug_path.write_bytes(debug_summary)

REFERENCE_CODE = RELEASE/'code'
CHILD_ENV = dict(os.environ)
CHILD_ENV['PYTHONPATH'] = str(REFERENCE_CODE)
subprocess.run([str(PYTHON),str(PACKAGE/'person2.py'),'--help'], cwd=PACKAGE, env=CHILD_ENV, check=True)
print('Verified v4 release:', RELEASE)
print('New debug summary reconstructed with its exact published hash.')
print('Encoder provenance verified; all four requested budgets validated.')


## 7. Backup settings
Automatic Drive transfer is disabled. Outputs remain on SOL and handoff ZIPs are created locally. No rclone setup is needed with this setting.


In [ ]:
def find_rclone():
    value = RCLONE_OVERRIDE or shutil.which('rclone')
    if not value:
        raise RuntimeError('Configure/load rclone in the Terminal, then set RCLONE_OVERRIDE to its absolute executable path.')
    return value

def backup_results():
    if not AUTO_BACKUP_TO_DRIVE:
        print('Drive backup disabled. Results are only local until you copy/download them.')
        return
    rc = find_rclone()
    dest = f'{DRIVE_REMOTE}:{DRIVE_FOLDER}/{ATTEMPT}/results'
    # copy does not delete destination files; incomplete checkpoint folders are excluded.
    subprocess.run([rc,'copy',str(WORK),dest,'--exclude','*.incomplete/**',
                    '--transfers','2','--checkers','4','--stats','30s'],check=True)
    print('Drive copy completed:',dest)

if AUTO_BACKUP_TO_DRIVE:
    rc=find_rclone()
    remotes=subprocess.check_output([rc,'listremotes'],text=True).splitlines()
    assert DRIVE_REMOTE+':' in remotes, 'Create the gdrive remote in the Terminal first.'
    # A small actual write tests the connection before GPU training starts.
    (WORK/'drive_connection_check.txt').write_text('Person 2 SOL backup connection check.\n')
    backup_results()


## 8. Execution and packaging helpers
Drive is off. Periodic checkpoints remain on SOL; each ZIP contains the final checkpoint, logs and provenance.


In [ ]:
import datetime, csv, zipfile

def model_work(model):
    return WORK/('qwen2_5_'+model.lower().replace('.','_'))

def common(model,scope):
    return ['--model',model,'--release',str(RELEASE),'--work',str(model_work(model)),'--scope',scope]

def run(command,*args):
    logs=WORK/'console';logs.mkdir(exist_ok=True)
    stamp=datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    log=logs/f'{stamp}_{command}.log'
    cmd=[str(PYTHON),'-u',str(PACKAGE/'person2.py'),command,*map(str,args)]
    with log.open('w') as output:
        process=subprocess.Popen(cmd,cwd=PACKAGE,env=CHILD_ENV,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        try:
            for line in process.stdout:
                print(line,end='');output.write(line);output.flush()
            status=process.wait()
        except KeyboardInterrupt:
            process.terminate();process.wait();raise
    if status:raise RuntimeError(f'{command} failed; see {log}. Dependent training has stopped.')
    try:backup_results()
    except Exception as error:print('COMMAND SUCCEEDED; DRIVE BACKUP FAILED. Retry backup_results():',error)

def verify_scope(model,scope):
    output=model_work(model)/'checks'/scope/'verification.json'
    if output.exists():
        print('Existing verification:',output,'; training will revalidate its fingerprints and counts.')
    else:run('verify',*common(model,scope),'--confirmed-by',WHO)

def completed(run_dir):
    summary=run_dir/'training_summary.json'
    return summary.is_file() and json.loads(summary.read_text()).get('complete') is True

def package_run(model,run_dir):
    if not completed(run_dir):raise ValueError('Cannot hand off an incomplete run')
    tag=model.lower().replace('.','_')
    dest=HANDOFFS/f'{ATTEMPT}_qwen2_5_{tag}_{run_dir.name}_handoff.zip'
    checks=[]
    with zipfile.ZipFile(dest,'w',zipfile.ZIP_DEFLATED) as z:
        for path in sorted(run_dir.rglob('*')):
            if path.is_file() and not any(part.endswith('.incomplete') or part.startswith('checkpoint-') for part in path.relative_to(run_dir).parts):
                arc='run/'+path.relative_to(run_dir).as_posix();z.write(path,arc)
                checks.append({'file':arc,'sha256':hashlib.sha256(path.read_bytes()).hexdigest()})
        for path,arc in [(model_work(model)/'prompt_template.json','prompt_template.json'),
                         (RELEASE/'configs/formatting_contract.json','person1_reference_formatting_contract.json'),
                         (RELEASE/'metadata/release_manifest.json','release_manifest.json'),
                         (RELEASE/'metadata/encoder_origin_manifest.json','encoder_origin_manifest.json')]:
            z.write(path,arc);checks.append({'file':arc,'sha256':hashlib.sha256(path.read_bytes()).hexdigest()})
        scope=json.loads((run_dir/'run_config.json').read_text())['rows']
        evidence=model_work(model)/'checks'/f'train_n{scope}'
        for path in sorted(evidence.glob('*')):
            if path.is_file():z.write(path,'verification/'+path.name)
        z.writestr('HANDOFF_CHECKSUMS.json',json.dumps(checks,indent=2))
        z.writestr('READ_FIRST.txt',
            'PRELIMINARY EXPERIMENT. Evaluation freeze and development/test protection are pending. '
            'See run/PRELIMINARY_STATUS.json; do not present as a finalized saturation point. '
            'Load the MODEL and REVISION from run/run_config.json, then run/final/adapter. '
            'Use the matching saved tokenizer and plain prompt. '
            'The formatting contract is Person 1 reference provenance, not an instruction to load 0.5B for a 1.5B adapter. '
            'Verification records show the tokenizer used and compatibility check. '
            'No benchmark accuracy is claimed by training logs. Only load trusted optimizer pickle checkpoints.')
    with zipfile.ZipFile(dest) as z:assert z.testzip() is None
    print('Saved handoff:',dest)
    if AUTO_BACKUP_TO_DRIVE:
        remote=f'{DRIVE_REMOTE}:{DRIVE_FOLDER}/{ATTEMPT}/handoffs/{dest.name}'
        try:
            subprocess.run([find_rclone(),'copyto',str(dest),remote,'--stats','30s'],check=True)
            print('Drive handoff copied:',remote)
        except Exception as error:
            print('Local ZIP saved; Drive copy failed. Retry package_run(model, run_dir):', error)
    return dest


## 9. Debug validation for both models
Fresh debug training and reload checks use the verified debug data and new summary. Debug adapters are not used to initialize the larger runs.


In [ ]:
for model in MODEL_SIZES:
    print(f'=== Debug validation: Qwen2.5-{model} ===')
    verify_scope(model,'debug_32')
    name=f'debug32_seed{SEED}'
    run_dir=model_work(model)/'runs'/name
    if run_dir.exists():
        if not completed(run_dir):raise RuntimeError(f'Incomplete debug run: {run_dir}; inspect or use a new attempt')
    else:
        run('train',*common(model,'debug_32'),'--run-name',name,'--seed',SEED,'--debug-epochs',10)
    run('reload',*common(model,'debug_32'),'--checkpoint',run_dir/'final')
print('Both debug models saved and reloaded. Selected training manifests will be verified next.')


## 10. Preliminary status
No team_readiness.json is created or required. The run records the outstanding evaluation and protection checks honestly.


In [ ]:
print('PRELIMINARY RUN: evaluation setup and development/test protection are not yet established.')
print('Results:', WORK)
print('Handoff ZIP folder:', HANDOFFS)


## 11. Train 16 runs and recover interrupted runs
One GPU, sequentially. Same data, independent base initialization per seed/budget/model. Automatic recovery requires a complete periodic checkpoint; no existing run is overwritten.


In [ ]:
def run_segments(model, name):
    root = model_work(model)/'runs'
    return sorted(p for p in root.glob(name+'*')
                  if p.is_dir() and (p.name == name or p.name.startswith(name+'_resume')))

def resolve_run(model, name):
    segments = run_segments(model, name)
    done = [p for p in segments if completed(p)]
    if done:
        return done[-1], None, True
    if not segments:
        return model_work(model)/'runs'/name, None, False
    required = ['adapter/adapter_model.safetensors', 'adapter/adapter_config.json',
                'training_state.pt', 'run_config.json', 'probe.pt', 'tokenizer/tokenizer_config.json']
    candidates = []
    for segment in segments:
        for cp in segment.glob('checkpoint-*'):
            if cp.name.split('-')[-1].isdigit() and all((cp/f).is_file() and (cp/f).stat().st_size for f in required):
                candidates.append(cp)
    if not candidates:
        raise RuntimeError(f'Interrupted run has no usable periodic checkpoint: {segments[-1]}. '
                           'Inspect the error and move that incomplete run to an archive folder before restarting.')
    checkpoint = max(candidates, key=lambda p: (int(p.name.split('-')[-1]), p.parent.name))
    counter = 1
    while (model_work(model)/'runs'/f'{name}_resume{counter:02d}').exists():
        counter += 1
    return model_work(model)/'runs'/f'{name}_resume{counter:02d}', checkpoint, False

for seed in SEEDS:
    for n in DATASET_SIZES:
        for model in MODEL_SIZES:
            print(f'=== {model}, {n} examples, one epoch, seed {seed} ===', flush=True)
            scope, name = f'train_n{n}', f'R{n}_seed{seed}'
            verify_scope(model, scope)
            run_dir, checkpoint, is_done = resolve_run(model, name)
            if not is_done:
                extra = ['--resume', checkpoint] if checkpoint else []
                if checkpoint: print('Resuming:', checkpoint, flush=True)
                run('train', *common(model, scope), '--run-name', run_dir.name,
                    '--seed', seed, '--preliminary', *extra)
            else:
                print('Already complete:', run_dir, flush=True)
            run('reload', *common(model, scope), '--checkpoint', run_dir/'final')
            package_run(model, run_dir)
print('All 16 main runs completed.')


## 12. Compare the saved TRAINING results
This table is training bookkeeping, not benchmark accuracy. Memory/time depend on architecture and hardware. Losses were measured on batches during training, not on a fixed final evaluation. Person 3 should separately evaluate each untrained base and each matching trained adapter using the same evaluation questions/settings.

In [ ]:
records = []
for seed in SEEDS:
    for n in DATASET_SIZES:
        for model in MODEL_SIZES:
            done = [p for p in run_segments(model, f'R{n}_seed{seed}') if completed(p)]
            if not done: continue
            run_dir = done[-1]
            summary = json.loads((run_dir/'training_summary.json').read_text())
            config = json.loads((run_dir/'run_config.json').read_text())
            records.append(dict(model=config['model'], model_revision=config['revision'],
                examples=n, seed=seed, epochs=config['epochs'],
                supervised_tokens=summary['processed_tokens'], updates=summary['optimizer_updates'],
                training_seconds=summary['training_seconds'],
                peak_allocated_GB_final_segment=summary['peak_allocated_GB'],
                experiment_status=config['experiment_status'], checkpoint=str(run_dir/'final'),
                manifest_sha256=config['manifest_sha256']))
if records:
    table = WORK/'seed_comparison_training_summary.csv'
    with table.open('w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=list(records[0]))
        writer.writeheader(); writer.writerows(records)
    for row in records: print(row)
    print('Saved:', table)
else:
    print('No completed main runs yet.')
print(f'{len(records)} / 16 complete. Training statistics are not benchmark accuracy.')


## Recovery and handoff
After an interruption, restart this notebook in a GPU allocation and run the cells in order. Completed main runs are reused; periodic recovery restores the existing trainer's optimizer and RNG state into a new `_resumeNN` folder. Exact data/code/config/environment checks remain active. Debug interruptions before validation finishes still require inspection.

The final CSV points Person 3 to the correct final checkpoint. All run segments remain on SOL. Periodic checkpoint cadence is 100 optimizer updates, so an interruption can lose up to the work since the latest saved checkpoint. These are independent seed replicates, not continuation from seed 42.

Original training engine: one epoch, LoRA rank16/alpha32/dropout0.05, 4-bit NF4, microbatch1/accumulation8, learning rate0.0001 with10-update warmup, response/EOS-only loss. No original trainer source was changed.

Validation here checks notebook/code syntax and queue logic. Actual GPU execution, file hashes, environment and tokenizer validation occur on SOL. The existing original release and matching updated manifest/formatting files are required; a screenshot cannot establish their contents. No remote SOL files were accessed during authoring.
